# Fraud Detection at Scale

Companion notebook for the [Fraud Detection at Scale lesson](https://ml-viz-ruby.vercel.app/courses/ml-in-practice/20-fraud-detection-at-scale).

We implement AUPRC evaluation under class imbalance, velocity feature computation, and a toy graph propagation to demonstrate guilt-by-association. Pure NumPy + Python stdlib.

> **To save your work:** File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')
rng = np.random.default_rng(99)

## 1 — Class imbalance: accuracy vs AUPRC

At 0.1% fraud rate, a trivial "not fraud" classifier scores 99.9% accuracy. We compute the Precision-Recall curve to show it's useless.

In [ ]:
# Simulate 10,000 transactions, 0.1% fraud
n = 10_000
fraud_rate = 0.001
labels = (rng.random(n) < fraud_rate).astype(int)
print(f"Total frauds: {labels.sum()} / {n} ({100*fraud_rate}% fraud rate)")

# Trivial model: predict "not fraud" for all
trivial_scores = np.zeros(n)
trivial_preds = np.zeros(n, dtype=int)
trivial_acc = (trivial_preds == labels).mean()
print(f"Trivial model accuracy: {trivial_acc:.4f}")

# A slightly better model: predict fraud with some signal
# Score = velocity feature proxy (random but slightly correlated with fraud)
scores = rng.uniform(0, 1, n)
scores[labels == 1] += 0.4  # fraud transactions score slightly higher
scores = np.clip(scores, 0, 1)

# Precision-Recall curve
def precision_recall_curve(labels, scores):
    thresholds = np.linspace(0, 1, 200)
    precs, recs = [], []
    for t in thresholds:
        pred = (scores >= t).astype(int)
        tp = ((pred == 1) & (labels == 1)).sum()
        fp = ((pred == 1) & (labels == 0)).sum()
        fn = ((pred == 0) & (labels == 1)).sum()
        precs.append(tp / (tp + fp + 1e-9))
        recs.append(tp / (tp + fn + 1e-9))
    return np.array(precs), np.array(recs)

precs, recs = precision_recall_curve(labels, scores)
auprc = np.trapz(precs[::-1], recs[::-1])

plt.figure(figsize=(7,4))
plt.plot(recs, precs, color='#6366f1', lw=2)
plt.axhline(fraud_rate, ls='--', color='gray', label=f'Random baseline ({fraud_rate:.3f})')
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title(f'Precision-Recall Curve (AUPRC={auprc:.4f})')
plt.legend(); plt.tight_layout(); plt.show()
print(f"AUPRC: {auprc:.4f}  (random baseline ≈ {fraud_rate:.4f})")

## 2 — Velocity features

Velocity features count activity in rolling windows — the strongest signal for payment fraud.

In [ ]:
# Simulate transaction log: (user_id, amount, timestamp_minutes)
n_users = 50
n_transactions = 500
user_ids  = rng.integers(0, n_users, n_transactions)
amounts   = rng.exponential(50, n_transactions)
times_min = np.sort(rng.uniform(0, 1440, n_transactions))  # 24 hours in minutes

def compute_velocity(user_ids, amounts, times_min, window_min=60):
    """For each transaction, count # transactions by same user in last `window_min` minutes."""
    n = len(user_ids)
    velocity = np.zeros(n, dtype=int)
    for i in range(n):
        t_start = times_min[i] - window_min
        # find all transactions by same user in [t_start, times_min[i])
        mask = (
            (user_ids[:i] == user_ids[i]) &
            (times_min[:i] >= t_start)
        )
        velocity[i] = mask.sum()
    return velocity

velocity_60min = compute_velocity(user_ids, amounts, times_min, window_min=60)
print(f"Max transactions by one user in any 60-min window: {velocity_60min.max()}")
print(f"Mean velocity: {velocity_60min.mean():.2f}")
print(f"Transactions with velocity > 5: {(velocity_60min > 5).sum()}")

## 3 — Graph propagation (guilt-by-association)

A simple message-passing step: if a node is fraudulent, propagate its fraud score to neighbors in the entity-sharing graph.

In [ ]:
# Toy entity graph: 10 accounts, some share devices
# Adjacency: account_i shares device with account_j
edges = [(0,1), (1,2), (3,4), (5,6), (6,7), (7,8)]  # pairs sharing devices
n_accounts = 10

adj = np.zeros((n_accounts, n_accounts))
for i,j in edges:
    adj[i,j] = adj[j,i] = 1

# Initial fraud scores (from feature model)
fraud_scores = np.array([0.9, 0.1, 0.1, 0.8, 0.1, 0.1, 0.2, 0.1, 0.1, 0.05])
# Accounts 0 and 3 are likely fraudsters

def graph_propagate(scores, adj, alpha=0.3, n_steps=2):
    """Propagate fraud scores through adjacency matrix for n_steps steps."""
    s = scores.copy()
    for _ in range(n_steps):
        neighbor_avg = adj @ s / (adj.sum(1) + 1e-9)
        s = (1 - alpha) * s + alpha * neighbor_avg  # blend own score + neighbor avg
    return s

propagated = graph_propagate(fraud_scores, adj)
print("Account | Initial score | After graph propagation")
for i in range(n_accounts):
    print(f"  {i:2d}    |  {fraud_scores[i]:.3f}        | {propagated[i]:.3f}  {'← elevated by neighbor' if propagated[i] > fraud_scores[i]+0.05 else ''}")

## ✏️ Your turn

**Exercise.** Implement `recall_at_fpr(labels, scores, target_fpr)`:
Given an array of true labels and model scores, find the recall achievable at the given False Positive Rate (FPR) target.

This is the standard operational metric: "what fraction of fraud do we catch at a 1% false alarm rate?

In [ ]:
def recall_at_fpr(labels, scores, target_fpr=0.01):
    """
    labels: 1D array, 1=fraud, 0=not fraud
    scores: 1D float array, higher = more likely fraud
    target_fpr: desired FPR threshold (e.g., 0.01 = 1%)
    Returns: recall at that FPR.
    """
    # TODO(you): sweep thresholds, find the one where FPR ≈ target_fpr,
    # and return the corresponding recall (TPR)
    return ...

r = recall_at_fpr(labels, scores, target_fpr=0.05)
print(f"Recall @ 5% FPR: {r:.4f}")

In [ ]:
# Assertion
def _ref(labels, scores, tfpr=0.05):
    thresholds = np.linspace(0,1,500)
    best_rec, best_diff = 0, 1
    for t in thresholds:
        pred = (scores >= t).astype(int)
        fp = ((pred==1)&(labels==0)).sum()
        tn = ((pred==0)&(labels==0)).sum()
        tp = ((pred==1)&(labels==1)).sum()
        fn = ((pred==0)&(labels==1)).sum()
        fpr = fp/(fp+tn+1e-9)
        if abs(fpr-tfpr) < best_diff:
            best_diff = abs(fpr-tfpr)
            best_rec = tp/(tp+fn+1e-9)
    return best_rec
ref = _ref(labels, scores)
res = recall_at_fpr(labels, scores)
assert abs(res - ref) < 0.1, f"Expected ~{ref:.3f}, got {res:.3f}"
print(f"✓ recall_at_fpr correct: {res:.4f}")

<details><summary>Solution</summary>

```python
def recall_at_fpr(labels, scores, target_fpr=0.01):
    thresholds = np.linspace(0, 1, 500)
    best_recall, best_diff = 0.0, 1.0
    for t in thresholds:
        pred = (scores >= t).astype(int)
        fp = ((pred == 1) & (labels == 0)).sum()
        tn = ((pred == 0) & (labels == 0)).sum()
        tp = ((pred == 1) & (labels == 1)).sum()
        fn = ((pred == 0) & (labels == 1)).sum()
        fpr = fp / (fp + tn + 1e-9)
        if abs(fpr - target_fpr) < best_diff:
            best_diff = abs(fpr - target_fpr)
            best_recall = tp / (tp + fn + 1e-9)
    return best_recall
```
</details>